In [2]:
!pip install -q transformers torch

In [1]:
import json
import random
import torch

# HuggingFace transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer

## Loading our saved subsets from Drive

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load datasets from your genai folder
with open("/content/drive/MyDrive/genai/eval_subset.json", "r") as f:
    eval_subset = json.load(f)

with open("/content/drive/MyDrive/genai/train_subset.json", "r") as f:
    train_subset = json.load(f)

print(f"Eval examples: {len(eval_subset)}")
print(f"Train examples: {len(train_subset)}")

Mounted at /content/drive
Eval examples: 500
Train examples: 5000


## Loading GPT-2

In [4]:
# Load GPT-2 small model and tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set padding token (important for generation)
tokenizer.pad_token = tokenizer.eos_token

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"Model loaded on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded on cpu


Prompt builder

In [5]:
def build_prompt(serialized_table, examples=None):
    """
    Build a prompt for GPT-2.

    If examples are provided → few-shot
    Otherwise → zero-shot
    """

    prompt = ""

    # few-shot examples
    if examples:
        for ex in examples:
            prompt += f"Table: {ex['serialized_table']}\n"
            prompt += f"Description: {ex['reference']}\n\n"

    # target table
    prompt += f"Table: {serialized_table}\n"
    prompt += "Description:"

    return prompt

Generation Function

In [14]:
def generate_text(prompt, max_new_tokens_val=200):
    """
    Generate text using GPT-2 given a prompt
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens_val,
        do_sample=True,        # enables randomness
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated[len(prompt):].strip()

Zero-shot example

In [15]:
example = eval_subset[0]

prompt = build_prompt(example["serialized_table"])

output = generate_text(prompt, max_new_tokens_val=200)

print("=== ZERO-SHOT ===")
print("Table:", example["serialized_table"])
print("\nGenerated:", output)
print("\nReference:", example["reference"])

=== ZERO-SHOT ===
Table: # : 76 | Took Office : Daniel Henry Chamberlain | Left Office : December 1, 1874

Generated: This is a brief listing of the occupations that led to Daniel Henry Chamberlain as a member of the House of Lords in 1874.
The title of this article refers to the man. He was one of the first members of the House of Lords who was to become a member of the British Parliament when he became President of the Council on Foreign Relations in 1854. The term has no official meaning since the 1844 Act.

Reference: Daniel Henry Chamberlain was the 76th Governor of South Carolina from 1874.


Few-shot

In [16]:
# Selecting 3 random training examples
few_shot_examples = random.sample(train_subset, 3)

example = eval_subset[1]

prompt = build_prompt(example["serialized_table"], few_shot_examples)

output = generate_text(prompt, max_new_tokens_val=200)

print("=== FEW-SHOT (3) ===")
print("Generated:", output)

=== FEW-SHOT (3) ===
Generated: In 2016–17, Gay is the oldest girl in school.

Table: Year : 2015 | Title : The Kids in Love | Role : Eliza

Description: Eliza is the eldest girl in school.

Table: Year : 2013 | Title : The Kids in Love | Role : Megan

Description: Megan is the youngest girl in school.

Table: Year : 2012 | Title : The Kids in Love | Role : Marisa

Description: Marisa is the oldest girl in school.

Table: Year : 2010 | Title : The Kids in Love | Role : Anna

Description: Anna is the youngest girl in school.

Table: Year : 2009 | Title : The Kids in Love | Role : Anna

Description: Anna is the youngest girl in school.

Table: Year : 2008 | Title : The Kids in Love | Role : Megan

Description: Megan is the youngest girl in school.

Table


In [17]:
for k in [5, 10]:
    few_shot_examples = random.sample(train_subset, k)

    example = eval_subset[2]
    prompt = build_prompt(example["serialized_table"], few_shot_examples)

    output = generate_text(prompt, max_new_tokens_val=200)

    print(f"\n=== FEW-SHOT ({k}) ===")
    print("Generated:", output)


=== FEW-SHOT (5) ===
Generated: Total: 119

Table: Total : 118

Description: Total : 117

Table: Total : 116

Description: Total : 115

Description: Total : 114

Description: Total : 113

Description: Total : 112

Description: Total : 111

Description: Total : 110

Description: Total : 109

Description: Total : 108

Description: Total : 107

Description: Total : 106

Description: Total : 105

Description: Total : 104

Description: Total : 103

Description: Total : 102

Description: Total : 101

Description: Total : 100

Description: Total : 99

Description: Total : 98

Description: Total : 97

Description: Total : 96

Description: Total : 95

Description: Total : 94

Description: Total : 93

Description: Total : 92

Description: Total : 91

=== FEW-SHOT (10) ===
Generated: In 2013, Venezuela had a total of 119. In 2012, the number was 120.

Table: Year : 2010 | Title : Côte d'Ivoire | Character : Jules Delorme

Description: In 2010, Jacques De Laet played the role of Jules Delorme in 

In [18]:
results = []

for i in range(10):  # small test first
    ex = eval_subset[i]

    prompt = build_prompt(ex["serialized_table"])
    output = generate_text(prompt, max_new_tokens_val=200)

    results.append({
        "table": ex["serialized_table"],
        "reference": ex["reference"],
        "generated": output
    })

print("Generated 10 examples")

Generated 10 examples


In [19]:
for i in range(len(results)):
    print(f"\n--- Example {i+1} ---")
    print("TABLE:\n", results[i]["table"][:200], "...")
    print("\nREF:", results[i]["reference"])
    print("\nGEN:", results[i]["generated"])


--- Example 1 ---
TABLE:
 # : 76 | Took Office : Daniel Henry Chamberlain | Left Office : December 1, 1874 ...

REF: Daniel Henry Chamberlain was the 76th Governor of South Carolina from 1874.

GEN: The office has been a fixture of the government since the mid-1800's.
TOTAL TEN YEARS: 5
CATEGORY FOR PERCENTAGES: 100.00
The office was used by the government for a number of purposes, including office furniture, uniforms, uniforms of the army, war uniforms, medical equipment and supplies, uniforms of the navy, the navy's staff and the navy's soldiers' quarters. The average salary of the office was $45,000. The average rate of pay for the government employees was 10% of that of the employees.
The average rate of pay for the government employees was 10% of that of the employees. YEAR NUMBER: 4.00
The average rate of pay for the government employees was $43,000. The average rate of pay for the government employees was 10% of that of the employees.
The average rate of pay for the government 

In [20]:
output_path = "/content/drive/MyDrive/genai/gpt2_results.json"
with open(output_path, "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved results to {output_path}")

Saved results to /content/drive/MyDrive/genai/gpt2_results.json
